# HCM0204 render ablation (no retraining)

Notebook này dùng model `point_cloud.ply` đã có trong Kaggle Input và render bốn biến thể: bilinear/bicubic × sharpen 0/0.3. Ảnh public được lưu Q97, JPEG 4:4:4, `optimize=True` để chọn cấu hình trước khi render submission cuối.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

import torch

SCENE = "HCM0204"
BRANCH = "mdd_render_variants"
MODEL_ITERATION = -1  # tự chọn iteration lớn nhất trong point_cloud
JPEG_QUALITY = 97
JPEG_SUBSAMPLING = 0  # 4:4:4
SHARPEN_SIGMA = 0.7
SHARPEN_AMOUNT = 0.3
VARIANT_NAMES = (
    "bilinear_sharp0",
    "bilinear_sharp0p3",
    "bicubic_sharp0",
    "bicubic_sharp0p3",
)

WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "gaussian-splatting"
CLEAN_ROOT = WORK_ROOT / "cleaned_render_ablation"
VARIANT_ROOT = WORK_ROOT / "render_variants"
EVAL_ROOT = WORK_ROOT / "render_variant_evaluation"

assert torch.cuda.is_available(), "Hãy bật GPU Accelerator trong Kaggle"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
input_root = Path("/kaggle/input")
scene_matches = [
    path
    for path in input_root.rglob(SCENE)
    if (path / "train" / "images").is_dir()
    and (path / "train" / "sparse" / "0" / "cameras.bin").is_file()
    and (path / "test" / "images").is_dir()
    and (path / "test" / "test_poses.csv").is_file()
]
model_matches = [
    path.parent
    for path in input_root.rglob("cfg_args")
    if path.parent.name == SCENE
    and (path.parent / "point_cloud").is_dir()
]

print("Public scene candidates:")
for path in scene_matches:
    print(" -", path)
print("Model candidates:")
for path in model_matches:
    print(" -", path)

assert len(scene_matches) == 1, (
    f"Cần đúng một public scene {SCENE}, tìm thấy {len(scene_matches)}"
)
assert len(model_matches) == 1, (
    f"Cần đúng một model {SCENE} có cfg_args và point_cloud, "
    f"tìm thấy {len(model_matches)}. Hãy Add Input dataset model_ver001."
)

SCENE_SOURCE = scene_matches[0]
PUBLIC_ROOT = SCENE_SOURCE.parent
MODEL_SCENE = model_matches[0]
MODEL_ROOT = MODEL_SCENE.parent
PLY_FILES = sorted((MODEL_SCENE / "point_cloud").rglob("point_cloud.ply"))
assert PLY_FILES, f"Không tìm thấy point_cloud.ply trong {MODEL_SCENE}"
print("Public root:", PUBLIC_ROOT)
print("Model root:", MODEL_ROOT)
print("PLY files:", *PLY_FILES, sep="\n - " )


In [ ]:
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            "https://github.com/fulx17/gaussian-splatting.git",
            str(REPO),
        ],
        check=True,
    )
else:
    assert (REPO / ".git").exists(), f"{REPO} không phải Git repository"
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True)

subprocess.run(
    [
        "git", "submodule", "update", "--init", "--recursive",
        "submodules/diff-gaussian-rasterization",
        "submodules/simple-knn",
        "submodules/fused-ssim",
    ],
    cwd=REPO,
    check=True,
)
os.chdir(REPO)
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
if shutil.which("colmap") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "colmap"], check=True)

subprocess.run(
    [sys.executable, "scripts/apply_improved_gs_rasterizer_patch.py"],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-build-isolation", "--no-cache-dir", "--force-reinstall",
        str(REPO / "submodules" / "diff-gaussian-rasterization"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        str(REPO / "submodules" / "simple-knn"),
        str(REPO / "submodules" / "fused-ssim"),
        "plyfile", "lpips",
    ],
    check=True,
)
print("Dependencies installed")


In [ ]:
os.chdir(REPO)
from preprocess import preprocess_scene, undistort_scene
from scene.colmap_loader import read_intrinsics_binary

CLEAN_SCENE = CLEAN_ROOT / SCENE
if not CLEAN_SCENE.exists():
    CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SCENE_SOURCE, CLEAN_SCENE)
    undistort_scene(CLEAN_SCENE)
    preprocess_scene(CLEAN_SCENE)
else:
    print("Sử dụng lại scene đã preprocess:", CLEAN_SCENE)

test_images = sorted(
    path for path in (CLEAN_SCENE / "test" / "images").iterdir()
    if path.is_file()
)
with open(CLEAN_SCENE / "test" / "test_poses.csv", newline="") as file:
    test_rows = list(csv.DictReader(file))
cameras = read_intrinsics_binary(
    str(CLEAN_SCENE / "train" / "sparse" / "0" / "cameras.bin")
)
assert len(test_images) == len(test_rows) == 60
assert all(camera.model == "PINHOLE" for camera in cameras.values())
print("Preprocess hoàn tất; test images:", len(test_images))


In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_render_variants.py", "-v"],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [sys.executable, "scripts/apply_improved_gs_rasterizer_patch.py", "--check-only"],
    cwd=REPO,
    check=True,
)
print("Render-variant smoke checks passed")


In [ ]:
if VARIANT_ROOT.exists():
    shutil.rmtree(VARIANT_ROOT)
VARIANT_ROOT.mkdir(parents=True)

render_cmd = [
    sys.executable, "render_scene.py",
    "--model_dir", str(MODEL_ROOT),
    "--input_dir", str(CLEAN_ROOT),
    "--image_dir", str(VARIANT_ROOT),
    "--orig_dir", str(PUBLIC_ROOT),
    "--scene_name", SCENE,
    "--iterations", str(MODEL_ITERATION),
    "--render_variants",
    "--variant_sharpen_amount", str(SHARPEN_AMOUNT),
    "--sharpen_sigma", str(SHARPEN_SIGMA),
    "--jpeg_quality", str(JPEG_QUALITY),
    "--jpeg_subsampling", str(JPEG_SUBSAMPLING),
    "--jpeg_optimize",
]
print("Running:", " ".join(render_cmd))
subprocess.run(render_cmd, cwd=REPO, check=True)

variant_sizes = {}
for variant in VARIANT_NAMES:
    scene_dir = VARIANT_ROOT / variant / SCENE
    files = sorted(path for path in scene_dir.iterdir() if path.is_file())
    assert len(files) == 60, f"{variant}: expected 60 files, got {len(files)}"
    total_bytes = sum(path.stat().st_size for path in files)
    variant_sizes[variant] = total_bytes
    estimate_434_mb = total_bytes / len(files) * 434 / 1_000_000
    print(
        f"{variant}: {total_bytes / 1_000_000:.2f} MB / 60 images; "
        f"estimated 434 images = {estimate_434_mb:.2f} MB"
    )


In [ ]:
if EVAL_ROOT.exists():
    shutil.rmtree(EVAL_ROOT)
EVAL_ROOT.mkdir(parents=True)

results = []
for variant in VARIANT_NAMES:
    variant_eval_root = EVAL_ROOT / variant
    eval_cmd = [
        sys.executable, "eval_scene.py",
        "--input_dir", str(CLEAN_ROOT),
        "--image_dir", str(VARIANT_ROOT / variant),
        "--eval_dir", str(variant_eval_root),
        "--scene_name", SCENE,
        "--psnr_max", "40",
    ]
    subprocess.run(eval_cmd, cwd=REPO, check=True)
    result_path = variant_eval_root / f"{SCENE}.json"
    with open(result_path, encoding="utf-8") as file:
        result = json.load(file)
    assert result["num_images"] == 60
    result["variant"] = variant
    result["size_mb_60"] = variant_sizes[variant] / 1_000_000
    result["estimated_mb_434"] = (
        variant_sizes[variant] / result["num_images"] * 434 / 1_000_000
    )
    results.append(result)

results.sort(key=lambda item: item["weighted_score"], reverse=True)
print("\nRanking:")
for rank, result in enumerate(results, 1):
    print(
        f"{rank}. {result['variant']}: score={result['weighted_score']:.6f}, "
        f"PSNR={result['PSNR']:.6f}, SSIM={result['SSIM']:.6f}, "
        f"LPIPS={result['LPIPS']:.6f}, est434={result['estimated_mb_434']:.1f}MB"
    )

BEST = results[0]
BEST_VARIANT = BEST["variant"]
summary_path = WORK_ROOT / f"{SCENE}_render_variant_results.json"
with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)
print("Best variant:", BEST_VARIANT)
print("Saved summary:", summary_path)


In [ ]:
best_scene_dir = VARIANT_ROOT / BEST_VARIANT / SCENE
best_zip = WORK_ROOT / f"{SCENE}_{BEST_VARIANT}_q{JPEG_QUALITY}_444.zip"
if best_zip.exists():
    best_zip.unlink()
shutil.make_archive(
    str(best_zip.with_suffix("")),
    "zip",
    root_dir=best_scene_dir,
)
print("Best render ZIP:", best_zip)
print(f"ZIP size: {best_zip.stat().st_size / 1_000_000:.2f} MB")
print(
    "Submission 434 ảnh: ưu tiên Q96 4:4:4 optimized; chỉ dùng Q97 nếu "
    "tổng file cuối dưới 335MB."
)
